self RAG

In self RAG what we do is that First we decide does we need retrival for the question:
* If not then generate answer
* If yes then we retrieve , after that we go with flow:
    * Is retrieval documents relevant if not then no answer found
    * If yes , then we generate from context
        * Then we check that is generated answer supported
            * If yes then we check is useful , if yes then we go to the end
            * If not supported then we go to revise answer then we check is supported and again go with flow
            * If supported but not useful then we go to the retrieve portion again

In [ ]:
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from pydantic import BaseModel
from typing import List, TypedDict
from langchain_core.messages import HumanMessage
from langchain_core.documents import Document

from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_chroma import Chroma
import os 
os.environ["HF_HOME"] = "E:\\huggingface_embedding_cache"
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    encode_kwargs={"normalize_embeddings": True},
)

In [ ]:
llm = ChatOpenAI(
    model='gpt-4.1-mini',
    base_url="https://openai-rg-nadeem.openai.azure.com/openai/v1",
)

In [ ]:
# ==========================================================
# CREATE VECTOR DATABASE
# ==========================================================
loader = TextLoader(
    "data.txt",
    encoding="utf-8"
)

documents = loader.load()

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_documents(documents)

persist_directory='./vectorStore'



vectorStore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="my_docs",
    persist_directory=persist_directory,
)

retriever = vectorStore.as_retriever(
    search_kwargs={"k":3}
)

In [ ]:
class State(TypedDict):
    question: str
    documents: list
    need_retrieval: bool
    web_results: list
    is_relevant: bool
    is_supportive: bool
    generation: str
    final_generation: str
    revise_answer: bool
    count_revise: int

In [ ]:
def retriever(state: State):
    document = vectorStore.similarity_search(query=state['question'], k=4)
    return{
        "documents": document
    }


In [ ]:
class NeedRetrieval(BaseModel):
    condition: bool

structured_llm = llm.with_structured_output(NeedRetrieval)
def need_retrieval(state: State):
    retrieve = False
    result = structured_llm.invoke(f"you have to answer with with 0 or 1 boolean that is this question general means that it can be answered with out retrieval from the private documents the question is {state['question']}")
    if result.condition:
        retrieve = True
    return {
       "need_retrieval": retrieve
    }

In [ ]:
def gen_or_ret(state: State):
    if state['need_retrieval']:
        return "retriever"
    return "generation"

In [ ]:
class Relevant(BaseModel):
    condition: bool

structured_relevant_llm = llm.with_structured_output(Relevant)

def relevant_node(state: State):
    result = structured_relevant_llm.invoke(f"you have to tell me that is the retrieved documents are relevant to answer the question tell me in 0 or 1 as it is a boolean variable the question is {state['question']} and the retrieved documents are {state['documents']}")
    return {
        "is_relevant": result.condition
    }

def noAns_conGen(state: State):
    if state['is_relevant']:
        return "generationFromContext"
    return "noAnswer"


In [ ]:
def noAnswer(state: State):
    result = llm.invoke(f"Listen you have to answer the user that no answer is found for you the question is {state['question']} and the context we got was {state['documents']} so you have to answer him properly and you can suggest him better questions or ask question that can be answered from context that we got.")
    return {
    "final_generation": result.content
    }

In [ ]:
def generationFromContext(state: State):
    result = llm.invoke(f"you have to generate the answer from the context properly focusing properly on the question and retrieved documents. The question is {state['question']} and the context is {state['documents']}")
    return{
        "generation": result.content
    }

In [ ]:
class Relevant(BaseModel):
    condition: bool

structured_support_llm = llm.with_structured_output(Relevant)
def isSupported(state: State):
    result = structured_support_llm.invoke(f"You have to properly check the generated answer and the question and check that is that answer supporting the question that was asked the question is {state['question']} and the generated answer is {state['generation']}")
    return {
    "is_supportive" : result.condition
    }

def rev_isUse_route(state: State): 
    if state['count_revise'] >= 3:
        return "noAnswer"
    elif state['is_supportive']:
        return "isUse"
    return "revise_answer"

def revise_answer(state: State):
    result = llm.invoke(f"We generated the answer from the documents that we retrieved but we are getting that the generated answer is not supportive to answer the question so I am providing you the context or documents and the question and the answer we got you have regenerate the answer that is properly supportive to the question and is from the context. Question is {state['question']} the context is {state['documents']} and the generated answer was {state['generation']}")
    return{
        "generation": result.content,
        "count_revise": state['count_revise'] + 1
    }

def isUse(state: State):
    pass
